# Stocks with ≥ 10,000% returns — drawdown and time

Standalone scan. It does **not** use V5 entry signals.

NSE data is stored as **one parquet per trading day** (`nse_cm_YYYYMMDD.parquet`), with every listed name in that file. A previous version scanned each file in isolation, so every symbol had **one bar** and nothing could pass the length filter. This notebook **stitches the full history per symbol** first.

**10,000% return** means close reaches **101×** entry close.

**Path:** buy at the expanding-min close; stop at the first later close ≥ 101× that low. Drawdown is only between those two dates.


In [ ]:
from pathlib import Path
import warnings, gc
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

from quant.config import DATA_DIR as QUANT_DATA_DIR, PARQUET_DATA_DIR, NIFTY50_PATH

DATA_DIR = PARQUET_DATA_DIR
OUT_DIR = QUANT_DATA_DIR / "results" / "hundred_baggers"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PANEL_PATH = OUT_DIR / "_panel_ohlc.parquet"
FEATURES_PATH = QUANT_DATA_DIR / "results" / "5x_turnaround_v5" / "features" / "stock_features.parquet"

MIN_RETURN = 100.0
TARGET_MULT = 1.0 + MIN_RETURN
MIN_BARS = 60
MIN_ENTRY_PRICE = 1.0

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 80)
pd.options.mode.copy_on_write = True

print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print(f"Target: +{MIN_RETURN:.0f}%  ({TARGET_MULT:.0f}x)")
print("NIFTY:", NIFTY50_PATH, "exists" if NIFTY50_PATH.exists() else "missing")
assert DATA_DIR.exists(), f"Missing parquet dir: {DATA_DIR}"


In [ ]:
def first_series(frame, name):
    s = frame[name]
    return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s

def normalize_ohlc(df, source_name):
    cols = {str(c).lower().strip(): c for c in df.columns}
    aliases = {
        "date": ["date", "datetime", "timestamp"],
        "symbol": ["symbol", "ticker", "security", "stock"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close", "adj close", "adj_close"],
        "volume": ["volume", "vol", "qty", "quantity", "total traded quantity"],
        "open": ["open"],
    }
    out = {}
    for target, names in aliases.items():
        found = next((cols[n] for n in names if n in cols), None)
        if found is not None:
            out[target] = found
    missing = [x for x in ["date", "high", "low", "close"] if x not in out]
    if missing:
        raise ValueError(f"{source_name}: missing {missing}; columns={list(df.columns)}")
    symbol = first_series(df, out["symbol"]) if "symbol" in out else Path(source_name).stem
    vol = pd.to_numeric(first_series(df, out["volume"]), errors="coerce") if "volume" in out else np.nan
    opn = pd.to_numeric(first_series(df, out["open"]), errors="coerce") if "open" in out else np.nan
    x = pd.DataFrame({
        "Date": pd.to_datetime(first_series(df, out["date"]), errors="coerce"),
        "Symbol": pd.Series(symbol, index=df.index).astype(str).str.upper().str.strip(),
        "Open": opn,
        "High": pd.to_numeric(first_series(df, out["high"]), errors="coerce"),
        "Low": pd.to_numeric(first_series(df, out["low"]), errors="coerce"),
        "Close": pd.to_numeric(first_series(df, out["close"]), errors="coerce"),
        "Volume": vol,
    })
    x = x.dropna(subset=["Date", "Close"])
    if getattr(x["Date"].dt, "tz", None) is not None:
        x["Date"] = x["Date"].dt.tz_convert("UTC").dt.tz_localize(None)
    x = x.loc[x["Close"] > 0]
    for c in ["Open", "High", "Low", "Close", "Volume"]:
        x[c] = x[c].astype("float32")
    x["Symbol"] = x["Symbol"].astype("string")
    return x

PANEL_SCHEMA = pa.schema([
    pa.field("Date", pa.timestamp("ns"), nullable=True),
    pa.field("Symbol", pa.large_string(), nullable=True),
    pa.field("Open", pa.float32(), nullable=True),
    pa.field("High", pa.float32(), nullable=True),
    pa.field("Low", pa.float32(), nullable=True),
    pa.field("Close", pa.float32(), nullable=True),
    pa.field("Volume", pa.float32(), nullable=True),
])

def max_dd_on_path(close, high, low):
    c = np.asarray(close, dtype=np.float64)
    h = np.asarray(high, dtype=np.float64)
    lo = np.asarray(low, dtype=np.float64)
    peak_c = np.maximum.accumulate(c)
    dd_c = float(np.nanmin(c / np.maximum(peak_c, 1e-12) - 1.0))
    peak_hl = np.maximum.accumulate(np.fmax(c, np.where(np.isfinite(h), h, c)))
    trough = np.where(np.isfinite(lo), lo, c)
    rel = trough / np.maximum(peak_hl, 1e-12) - 1.0
    dd_l = float(np.nanmin(rel))
    worst_i = int(np.nanargmin(rel))
    return dd_c, dd_l, worst_i

def first_target_run(dates, close, high, low, symbol):
    c = np.asarray(close, dtype=np.float64)
    n = len(c)
    if n < MIN_BARS:
        return None, 0.0, n
    rmin = np.inf
    rmin_i = -1
    max_mult = 0.0
    hit = None
    for i in range(n):
        px = c[i]
        if not np.isfinite(px) or px <= 0:
            continue
        if px < rmin:
            rmin = px
            rmin_i = i
        if rmin_i >= 0 and rmin > 0:
            max_mult = max(max_mult, px / rmin)
        if hit is None and rmin_i >= 0 and px >= rmin * TARGET_MULT:
            sl = slice(rmin_i, i + 1)
            dd_c, dd_l, w_i = max_dd_on_path(c[sl], high[sl], low[sl])
            d0 = pd.Timestamp(dates[rmin_i])
            d1 = pd.Timestamp(dates[i])
            hit = {
                "Symbol": symbol,
                "EntryDate": d0,
                "HitDate": d1,
                "EntryClose": float(rmin),
                "HitClose": float(px),
                "Return": float(px / rmin - 1.0),
                "ReturnPct": float((px / rmin - 1.0) * 100.0),
                "Multiple": float(px / rmin),
                "TradingDays": int(i - rmin_i),
                "CalendarDays": int((d1 - d0).days),
                "Years": float((d1 - d0).days / 365.25),
                "MaxDrawdownClose": dd_c,
                "MaxDrawdownLow": dd_l,
                "WorstDDDate": pd.Timestamp(dates[rmin_i + w_i]),
                "BarsInSample": n,
                "SampleStart": pd.Timestamp(dates[0]),
                "SampleEnd": pd.Timestamp(dates[-1]),
                "MaxMultipleFromExpandingMin": float(max_mult),
                "LowEntryFlag": bool(rmin < MIN_ENTRY_PRICE),
                "EntryIndex": int(rmin_i),
            }
    if hit is not None:
        hit["MaxMultipleFromExpandingMin"] = float(max_mult)
    return hit, float(max_mult), n


In [ ]:
stock_files = sorted(
    p for p in DATA_DIR.rglob("*.parquet")
    if "results" not in p.parts and "indices" not in p.parts
)
print("Stock parquet files:", len(stock_files))
if stock_files:
    sample = pd.read_parquet(stock_files[0])
    print("Sample file:", stock_files[0].name, "rows:", len(sample), "cols:", list(sample.columns)[:12])
    print("Unique dates in sample file:", pd.to_datetime(sample.iloc[:, 0], errors="coerce").nunique() if len(sample) else 0)
    del sample

panel_src = None
if FEATURES_PATH.exists():
    panel_src = "features"
    print("Using existing feature panel:", FEATURES_PATH)
elif not stock_files:
    raise FileNotFoundError(f"No stock parquet under {DATA_DIR}")
else:
    if PANEL_PATH.exists():
        PANEL_PATH.unlink()
    writer = None
    n_ok = n_err = 0
    peek_dates = []
    for i, f in enumerate(stock_files, 1):
        try:
            raw = pd.read_parquet(f)
            x = normalize_ohlc(raw, f.name)
            del raw
            if x.empty:
                n_err += 1
                continue
            if len(peek_dates) < 5:
                peek_dates.append((f.name, int(x["Date"].nunique()), len(x), int(x["Symbol"].nunique())))
            table = pa.Table.from_pandas(x[["Date", "Symbol", "Open", "High", "Low", "Close", "Volume"]], schema=PANEL_SCHEMA, preserve_index=False)
            del x
            if writer is None:
                writer = pq.ParquetWriter(PANEL_PATH, PANEL_SCHEMA, compression="zstd")
            writer.write_table(table)
            del table
            n_ok += 1
            if i % 250 == 0:
                gc.collect()
                print(f"stitched {i}/{len(stock_files)} files")
        except Exception as e:
            n_err += 1
            if n_err <= 8:
                print("skip", f.name, e)
    if writer is not None:
        writer.close()
    gc.collect()
    print("Stitched files OK:", n_ok, "errors:", n_err)
    print("Per-file layout (name, unique_dates, rows, symbols):", peek_dates)
    panel_src = "stitched"

def read_panel_cols(path):
    have = set(pq.ParquetFile(path).schema.names)
    cols = [c for c in ["Date", "Symbol", "Open", "High", "Low", "Close", "Volume"] if c in have]
    return pd.read_parquet(path, columns=cols)

if panel_src == "features":
    panel = read_panel_cols(FEATURES_PATH)
else:
    panel = read_panel_cols(PANEL_PATH)
for c in ["Open", "Volume"]:
    if c not in panel.columns:
        panel[c] = np.nan

panel["Date"] = pd.to_datetime(panel["Date"])
panel["Symbol"] = panel["Symbol"].astype("string")
print("Panel rows:", len(panel), "symbols:", panel["Symbol"].nunique(),
      "dates:", panel["Date"].min(), "→", panel["Date"].max())
print("Median bars per symbol:", float(panel.groupby("Symbol", observed=True).size().median()))


In [ ]:
rows = []
mult_rows = []
for sym, g in panel.groupby("Symbol", sort=False, observed=True):
    g = g.sort_values("Date").drop_duplicates("Date", keep="last")
    rec, max_mult, n = first_target_run(
        g["Date"].to_numpy(),
        g["Close"].to_numpy(),
        g["High"].to_numpy(),
        g["Low"].to_numpy(),
        str(sym),
    )
    mult_rows.append({"Symbol": str(sym), "Bars": n, "MaxMultipleFromExpandingMin": max_mult})
    if rec is not None:
        rows.append(rec)

gc.collect()

mults = pd.DataFrame(mult_rows)
hits = pd.DataFrame(rows)
print("Symbols scanned:", len(mults))
print("Max multiple percentiles:")
print(mults["MaxMultipleFromExpandingMin"].replace(0, np.nan).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
for m, label in [(2, "100%"), (11, "1,000%"), (51, "5,000%"), (101, "10,000%")]:
    n = int((mults["MaxMultipleFromExpandingMin"] >= m).sum())
    print(f"  names with ≥ {label} from expanding-min close ({m:.0f}x): {n}")
print("Hits at +10,000%:", len(hits))
mults.sort_values("MaxMultipleFromExpandingMin", ascending=False).head(30).to_csv(OUT_DIR / "top_multiples_all_names.csv", index=False)


## Results

Only names that actually printed a +10,000% close from the expanding-min entry. Sorted by calendar days.


In [ ]:
if hits.empty:
    print("Still no +10,000% names. Inspect top multiples CSV — if 10x/50x counts are large but 101x is 0, the data simply never reached 101x from a prior low.")
    top = mults.sort_values("MaxMultipleFromExpandingMin", ascending=False).head(40)
    display(top)
else:
    hits = hits.sort_values(["CalendarDays", "MaxDrawdownLow"]).reset_index(drop=True)
    hits.to_csv(OUT_DIR / "stocks_10000pct_from_expanding_min.csv", index=False)
    hits.to_parquet(OUT_DIR / "stocks_10000pct_from_expanding_min.parquet", index=False)
    view = hits[[
        "Symbol", "EntryDate", "HitDate", "EntryClose", "HitClose",
        "ReturnPct", "Multiple", "TradingDays", "CalendarDays", "Years",
        "MaxDrawdownClose", "MaxDrawdownLow", "WorstDDDate", "LowEntryFlag",
        "MaxMultipleFromExpandingMin",
    ]].copy()
    show = view.copy()
    show["MaxDrawdownClose"] = show["MaxDrawdownClose"].map(lambda v: f"{v:.2%}")
    show["MaxDrawdownLow"] = show["MaxDrawdownLow"].map(lambda v: f"{v:.2%}")
    show["ReturnPct"] = show["ReturnPct"].map(lambda v: f"{v:,.0f}%")
    display(show)
    print("Count:", len(hits))
    print("Median calendar days:", float(hits["CalendarDays"].median()))
    print("Median trading days:", float(hits["TradingDays"].median()))
    print("Median max DD (close):", f"{hits['MaxDrawdownClose'].median():.2%}")
    print("Median max DD (low):", f"{hits['MaxDrawdownLow'].median():.2%}")
    print("Worst max DD (low):", f"{hits['MaxDrawdownLow'].min():.2%}",
          hits.loc[hits["MaxDrawdownLow"].idxmin(), "Symbol"])
    print("Fastest calendar days:", int(hits["CalendarDays"].min()),
          hits.loc[hits["CalendarDays"].idxmin(), "Symbol"])


In [ ]:
if not hits.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].hist(hits["CalendarDays"], bins=min(40, max(5, len(hits))), color="steelblue", edgecolor="white")
    axes[0].set_title("Calendar days to +10,000%")
    axes[1].hist(hits["MaxDrawdownLow"] * 100, bins=min(40, max(5, len(hits))), color="indianred", edgecolor="white")
    axes[1].set_title("Max DD on the path (lows)")
    axes[2].scatter(hits["CalendarDays"], hits["MaxDrawdownLow"] * 100, alpha=0.7)
    axes[2].set_xlabel("Calendar days")
    axes[2].set_ylabel("Max DD %")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "days_and_drawdown.png", dpi=120)
    plt.show()
    plt.close(fig)
    print("CSV:", OUT_DIR / "stocks_10000pct_from_expanding_min.csv")


## How to read a row

Bought `EntryClose` on `EntryDate` (lowest close up to then). First +10,000% close is `HitDate`. `MaxDrawdownLow` / `MaxDrawdownClose` is the worst peak-to-trough **on that path**.


## Pre-rally pattern (history available at the trough only)

`EntryDate` is a **new expanding-min close** by construction, so “it was at a low” is not a finding. The question is whether hundred-baggers **got there the same way**, and whether that path differs from **new lows that never became 101×**.

All features use bars `≤ EntryIndex`. Nothing after the trough is used to describe the setup.


In [ ]:
def rolling_mean(a, n):
    a = np.asarray(a, dtype=np.float64)
    if len(a) < n or n <= 0:
        return np.nan
    return float(np.nanmean(a[-n:]))

def rolling_std(a, n):
    a = np.asarray(a, dtype=np.float64)
    if len(a) < n:
        return np.nan
    x = a[-n:]
    x = x[np.isfinite(x)]
    if len(x) < max(5, n // 4):
        return np.nan
    return float(np.std(x, ddof=1))

def sma(a, n):
    a = np.asarray(a, dtype=np.float64)
    if len(a) < n:
        return np.nan
    return float(np.nanmean(a[-n:]))

def pre_trough_features(dates, o, h, l, c, v, i):
    """Features at index i using only [:i+1]."""
    dates = pd.to_datetime(np.asarray(dates))
    o = np.asarray(o, dtype=np.float64)
    h = np.asarray(h, dtype=np.float64)
    l = np.asarray(l, dtype=np.float64)
    c = np.asarray(c, dtype=np.float64)
    v = np.asarray(v, dtype=np.float64)
    n = i + 1
    if n < 5:
        return {}
    c0, h0, l0, v0 = c[:n], h[:n], l[:n], v[:n]
    px = c0[-1]
    rng = np.where(np.isfinite(h0) & np.isfinite(l0), h0 - l0, np.nan)
    # Prior peak strictly before the trough bar
    if n >= 2:
        peak_i = int(np.nanargmax(c0[:-1]))
        prior_peak = float(c0[peak_i])
        days_since_peak = int((dates[i] - dates[peak_i]).days)
        bars_since_peak = int(i - peak_i)
        dd_from_peak = px / prior_peak - 1.0 if prior_peak > 0 else np.nan
    else:
        peak_i = 0
        prior_peak = px
        days_since_peak = 0
        bars_since_peak = 0
        dd_from_peak = 0.0
    vol20 = rolling_std(np.diff(np.log(np.clip(c0, 1e-12, None))), 20)
    vol60 = rolling_std(np.diff(np.log(np.clip(c0, 1e-12, None))), 60)
    vol120 = rolling_std(np.diff(np.log(np.clip(c0, 1e-12, None))), 120)
    atrp20 = rolling_mean(rng / np.clip(c0, 1e-12, None), 20)
    atrp120 = rolling_mean(rng / np.clip(c0, 1e-12, None), 120)
    v20 = rolling_mean(v0, 20)
    v120 = rolling_mean(v0, 120)
    v_climax = (np.nanmax(v0[-20:]) / v120) if n >= 20 and np.isfinite(v120) and v120 > 0 else np.nan
    sma20 = sma(c0, 20)
    sma50 = sma(c0, 50)
    sma200 = sma(c0, 200)
    sma200_prev = sma(c0[:-20], 200) if n > 220 else np.nan
    # Waterfall: share of last 120 days that made a 20-day low
    waterfall = np.nan
    if n >= 40:
        last = c0[-120:] if n >= 120 else c0
        hits_ll = 0
        for k in range(20, len(last)):
            if last[k] <= np.nanmin(last[k-20:k]) + 1e-12:
                hits_ll += 1
        waterfall = hits_ll / max(len(last) - 20, 1)
    # Base tightness last 60 vs 252 range
    rng60 = (np.nanmax(h0[-60:]) / np.nanmin(l0[-60:]) - 1.0) if n >= 60 else np.nan
    rng252 = (np.nanmax(h0[-252:]) / np.nanmin(l0[-252:]) - 1.0) if n >= 252 else np.nan
    ret20 = px / c0[-21] - 1.0 if n > 21 else np.nan
    ret60 = px / c0[-61] - 1.0 if n > 61 else np.nan
    ret252 = px / c0[-253] - 1.0 if n > 253 else np.nan
    return {
        "TroughDate": pd.Timestamp(dates[i]),
        "TroughClose": float(px),
        "BarsBefore": int(i),
        "PriorPeakClose": prior_peak,
        "DaysSincePriorPeak": days_since_peak,
        "BarsSincePriorPeak": bars_since_peak,
        "DrawdownFromPriorPeak": dd_from_peak,
        "Vol20": vol20,
        "Vol60": vol60,
        "Vol120": vol120,
        "VolCompression": (vol20 / vol120) if vol120 and np.isfinite(vol20) and vol120 > 0 else np.nan,
        "ATRp20": atrp20,
        "ATRp120": atrp120,
        "RangeCompression": (atrp20 / atrp120) if atrp120 and atrp120 > 0 else np.nan,
        "Volume20": v20,
        "Volume120": v120,
        "VolumeDryUp": (v20 / v120) if v120 and v120 > 0 else np.nan,
        "VolumeClimax20": v_climax,
        "CloseVsSMA20": px / sma20 - 1.0 if sma20 else np.nan,
        "CloseVsSMA50": px / sma50 - 1.0 if sma50 else np.nan,
        "CloseVsSMA200": px / sma200 - 1.0 if sma200 else np.nan,
        "SMA200Slope20": (sma200 / sma200_prev - 1.0) if sma200_prev else np.nan,
        "Range60": rng60,
        "Range252": rng252,
        "Return20": ret20,
        "Return60": ret60,
        "Return252": ret252,
        "WaterfallLowRate120": waterfall,
        "TroughMonth": int(pd.Timestamp(dates[i]).month),
        "TroughYear": int(pd.Timestamp(dates[i]).year),
        "TroughWeekday": int(pd.Timestamp(dates[i]).dayofweek),
        "ListingToTroughDays": int((dates[i] - dates[0]).days),
    }

def load_nifty_close():
    if not NIFTY50_PATH.exists():
        return None
    n = pd.read_parquet(NIFTY50_PATH)
    cols = {str(c).lower().strip(): c for c in n.columns}
    dcol = next((cols[k] for k in ["date", "datetime", "timestamp"] if k in cols), None)
    ccol = next((cols[k] for k in ["close", "adj close", "adj_close"] if k in cols), None)
    if dcol is None or ccol is None:
        return None
    out = pd.DataFrame({
        "Date": pd.to_datetime(n[dcol], errors="coerce"),
        "NiftyClose": pd.to_numeric(n[ccol], errors="coerce"),
    }).dropna()
    if getattr(out["Date"].dt, "tz", None) is not None:
        out["Date"] = out["Date"].dt.tz_convert("UTC").dt.tz_localize(None)
    out = out.sort_values("Date").drop_duplicates("Date")
    out["NiftySMA200"] = out["NiftyClose"].rolling(200, min_periods=200).mean()
    out["NiftyRet60"] = out["NiftyClose"] / out["NiftyClose"].shift(60) - 1.0
    out["NiftyRet252"] = out["NiftyClose"] / out["NiftyClose"].shift(252) - 1.0
    return out.set_index("Date")

nifty = load_nifty_close()
print("Nifty rows:", 0 if nifty is None else len(nifty))


In [ ]:
# Rebuild per-symbol arrays once for pattern + controls (panel still in memory).
if "panel" not in globals() or panel is None or len(panel) == 0:
    raise RuntimeError("Panel was dropped. Re-run the stitch cell.")

hit_syms = set(hits["Symbol"].astype(str)) if hits is not None and len(hits) else set()
feat_hits = []
feat_ctrl = []
rng_ctrl = np.random.default_rng(42)

for sym, g in panel.groupby("Symbol", sort=False, observed=True):
    g = g.sort_values("Date").drop_duplicates("Date", keep="last")
    dates = g["Date"].to_numpy()
    o = g["Open"].to_numpy() if "Open" in g.columns else np.full(len(g), np.nan)
    h = g["High"].to_numpy()
    l = g["Low"].to_numpy()
    c = g["Close"].to_numpy(dtype=np.float64)
    v = g["Volume"].to_numpy() if "Volume" in g.columns else np.full(len(g), np.nan)
    n = len(c)
    if n < MIN_BARS:
        continue
    rec, max_mult, _ = first_target_run(dates, c, h, l, str(sym))
    if rec is not None:
        i = int(rec["EntryIndex"])
        f = pre_trough_features(dates, o, h, l, c, v, i)
        f["Symbol"] = str(sym)
        f["Group"] = "hit_101x"
        f["LaterMaxMultiple"] = float(max_mult)
        feat_hits.append(f)
        continue
    # Controls: up to two new expanding-mins that never reached 101x
    rmin = np.inf
    low_idx = []
    for i in range(n):
        px = c[i]
        if not np.isfinite(px) or px <= 0:
            continue
        if px < rmin:
            rmin = px
            if i >= 60:
                low_idx.append(i)
    if not low_idx:
        continue
    pick = []
    pick.append(low_idx[-1])
    if len(low_idx) >= 3:
        pick.append(low_idx[len(low_idx) // 2])
    pick = list(dict.fromkeys(pick))
    for i in pick:
        f = pre_trough_features(dates, o, h, l, c, v, i)
        f["Symbol"] = str(sym)
        f["Group"] = "failed_new_low"
        f["LaterMaxMultiple"] = float(max_mult)
        feat_ctrl.append(f)

pre = pd.DataFrame(feat_hits)
ctrl = pd.DataFrame(feat_ctrl)
print("Hit troughs with features:", len(pre), "control new-lows:", len(ctrl))

def attach_nifty(df):
    if nifty is None or df.empty or "TroughDate" not in df.columns:
        return df
    x = df.copy()
    x["TroughDate"] = pd.to_datetime(x["TroughDate"])
    n = nifty.reset_index().rename(columns={"Date": "TroughDate"})
    x = pd.merge_asof(
        x.sort_values("TroughDate"),
        n.sort_values("TroughDate"),
        on="TroughDate",
        direction="backward",
    )
    x["NiftyAbove200"] = x["NiftyClose"] > x["NiftySMA200"]
    return x

pre = attach_nifty(pre)
ctrl = attach_nifty(ctrl)
pre.to_csv(OUT_DIR / "pre_rally_features_hits.csv", index=False)
ctrl.to_csv(OUT_DIR / "pre_rally_features_controls.csv", index=False)


In [ ]:
NUM_COLS = [
    "DaysSincePriorPeak", "BarsSincePriorPeak", "DrawdownFromPriorPeak",
    "Vol20", "Vol60", "VolCompression", "ATRp20", "RangeCompression",
    "VolumeDryUp", "VolumeClimax20",
    "CloseVsSMA20", "CloseVsSMA50", "CloseVsSMA200", "SMA200Slope20",
    "Range60", "Range252", "Return20", "Return60", "Return252",
    "WaterfallLowRate120", "ListingToTroughDays", "TroughClose",
    "NiftyRet60", "NiftyRet252",
]
NUM_COLS = [c for c in NUM_COLS if c in pre.columns or c in ctrl.columns]

def summarize_group(df, name):
    rows = []
    for c in NUM_COLS:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        rows.append({
            "Feature": c,
            "Group": name,
            "N": int(s.notna().sum()),
            "Median": float(s.median()),
            "P25": float(s.quantile(0.25)),
            "P75": float(s.quantile(0.75)),
            "Mean": float(s.mean()),
        })
    return pd.DataFrame(rows)

cmp = pd.concat([summarize_group(pre, "hit_101x"), summarize_group(ctrl, "failed_new_low")], ignore_index=True)
wide = cmp.pivot(index="Feature", columns="Group", values=["Median", "P25", "P75", "N"])
# flatten
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()
if "Median_hit_101x" in wide.columns and "Median_failed_new_low" in wide.columns:
    wide["HitMinusControlMedian"] = wide["Median_hit_101x"] - wide["Median_failed_new_low"]
display(wide.sort_values("Feature"))
wide.to_csv(OUT_DIR / "pre_rally_hit_vs_control.csv", index=False)

print("\nShare of hit troughs vs failed new-lows")
rules = [
    ("Drawdown from prior peak ≤ −70%", "DrawdownFromPriorPeak", lambda s: s <= -0.70),
    ("Drawdown from prior peak ≤ −90%", "DrawdownFromPriorPeak", lambda s: s <= -0.90),
    ("≥ 2 years since prior peak", "DaysSincePriorPeak", lambda s: s >= 365 * 2),
    ("≥ 5 years since prior peak", "DaysSincePriorPeak", lambda s: s >= 365 * 5),
    ("Below SMA200", "CloseVsSMA200", lambda s: s < 0),
    ("SMA200 still falling (20d)", "SMA200Slope20", lambda s: s < 0),
    ("Vol compressed (20/120 < 0.7)", "VolCompression", lambda s: s < 0.70),
    ("Range compressed (ATR20/ATR120 < 0.7)", "RangeCompression", lambda s: s < 0.70),
    ("Volume dry-up (20/120 < 0.7)", "VolumeDryUp", lambda s: s < 0.70),
    ("Volume climax (≥ 3× 120d avg in last 20d)", "VolumeClimax20", lambda s: s >= 3.0),
    ("12m return ≤ −50%", "Return252", lambda s: s <= -0.50),
    ("Waterfall (new 20d lows often)", "WaterfallLowRate120", lambda s: s >= 0.15),
    ("Penny-ish trough close < ₹5", "TroughClose", lambda s: s < 5),
    ("Trough close < ₹20", "TroughClose", lambda s: s < 20),
]
if nifty is not None and "NiftyAbove200" in pre.columns:
    rules.append(("Nifty below SMA200", "NiftyAbove200", lambda s: s == False))
    rules.append(("Nifty 12m return < 0", "NiftyRet252", lambda s: s < 0))

rule_rows = []
for label, col, fn in rules:
    row = {"Rule": label}
    for name, df in [("hit_101x", pre), ("failed_new_low", ctrl)]:
        if col not in df.columns or df.empty:
            row[name] = np.nan
            continue
        s = df[col]
        m = fn(s.dropna()) if s.notna().any() else pd.Series(dtype=float)
        # fn on series of remaining
        if s.notna().any():
            row[name] = float(fn(s).mean()) if s.dtype != bool else float(fn(s.fillna(False)).mean())
            try:
                row[name] = float(fn(s).mean())
            except Exception:
                row[name] = float(fn(pd.to_numeric(s, errors="coerce")).mean())
        else:
            row[name] = np.nan
    rule_rows.append(row)
rule_df = pd.DataFrame(rule_rows)
display(rule_df)
rule_df.to_csv(OUT_DIR / "pre_rally_rule_rates.csv", index=False)


In [ ]:
if pre.empty:
    print("No hit features. Re-run the scan until hits is non-empty.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    pairs = [
        ("DrawdownFromPriorPeak", "Drawdown from prior peak"),
        ("DaysSincePriorPeak", "Days since prior peak"),
        ("VolCompression", "Vol20 / Vol120"),
        ("VolumeDryUp", "Volume20 / Volume120"),
        ("CloseVsSMA200", "Close vs SMA200"),
        ("Return252", "Trailing 12m return"),
    ]
    for ax, (col, title) in zip(axes.ravel(), pairs):
        if col in pre.columns:
            ax.hist(pd.to_numeric(pre[col], errors="coerce").dropna(), bins=20, alpha=0.7, label="101x hits", color="steelblue")
        if col in ctrl.columns and len(ctrl):
            ax.hist(pd.to_numeric(ctrl[col], errors="coerce").dropna(), bins=20, alpha=0.35, label="failed new lows", color="gray")
        ax.set_title(title)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "pre_rally_distributions.png", dpi=120)
    plt.show()
    plt.close(fig)

    print("Trough year counts (hits):")
    display(pre["TroughYear"].value_counts().sort_index())
    print("Trough month counts (hits):")
    display(pre["TroughMonth"].value_counts().sort_index())

    # Shared-pattern writeup from medians — descriptive, not a trading rule.
    def med(df, col):
        return float(pd.to_numeric(df[col], errors="coerce").median()) if col in df.columns and len(df) else np.nan

    lines = []
    lines.append(f"N hundred-bagger troughs: {len(pre)}; N failed new-lows: {len(ctrl)}")
    lines.append(
        f"Hits: median decline from prior peak {med(pre,'DrawdownFromPriorPeak'):.1%} "
        f"over {med(pre,'DaysSincePriorPeak'):.0f} calendar days "
        f"({med(pre,'BarsSincePriorPeak'):.0f} sessions)."
    )
    if len(ctrl):
        lines.append(
            f"Failed new-lows: median decline {med(ctrl,'DrawdownFromPriorPeak'):.1%} "
            f"over {med(ctrl,'DaysSincePriorPeak'):.0f} days."
        )
        def lift(col, hit_dir="lower"):
            h, c = med(pre, col), med(ctrl, col)
            return h, c
        h, c = lift("DrawdownFromPriorPeak")
        if pd.notna(h) and pd.notna(c) and h < c:
            lines.append("Hits typically fell farther from the prior peak than ordinary new lows.")
        h, c = lift("DaysSincePriorPeak")
        if pd.notna(h) and pd.notna(c) and h > c:
            lines.append("Hits typically spent longer declining / washing out before the trough.")
        h, c = lift("VolumeDryUp")
        if pd.notna(h) and pd.notna(c) and h < c:
            lines.append("Hits more often showed volume dry-up into the low (20d volume vs 120d).")
        elif pd.notna(h) and pd.notna(c) and h > c:
            lines.append("Hits more often showed heavier volume into the low (possible capitulation), not dry-up.")
        h, c = lift("VolCompression")
        if pd.notna(h) and pd.notna(c) and h < c:
            lines.append("Hits more often compressed volatility into the trough.")
        elif pd.notna(h) and pd.notna(c) and h > c:
            lines.append("Hits more often still had elevated volatility at the trough (crisis/waterfall), not a tight base.")
        h, c = lift("TroughClose")
        if pd.notna(h) and pd.notna(c) and h < c:
            lines.append("Hits printed the 101× from a lower rupee price than typical failed new-lows (small-price / deep-distress names).")
    if "NiftyAbove200" in pre.columns and pre["NiftyAbove200"].notna().any():
        share = 1.0 - float(pre["NiftyAbove200"].mean())
        lines.append(f"Share of hit troughs with Nifty below SMA200: {share:.1%}.")
    lines.append("Do not treat this as an entry model. It is a description of how 101× names looked at a sample low vs other sample lows.")
    print("\n".join(lines))
    (OUT_DIR / "pre_rally_pattern_summary.txt").write_text("\n".join(lines) + "\n")

del panel
gc.collect()
print("Wrote", OUT_DIR)
